In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import keras

In [ ]:
dataset, dataset_info = tfds.load("malaria", with_info = True, split = "train", as_supervised = True)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

In [ ]:
def split(dataset, train_size, val_size):
    dataset_size = len(dataset)
    train_dataset = dataset.take(int(dataset_size*train_size))
    val_test_dataset = dataset.skip(int(dataset_size*train_size))
    val_dataset = val_test_dataset.take(int(dataset_size*val_size))
    test_dataset = val_test_dataset.skip(int(dataset_size*val_size))
    return train_dataset, val_dataset, test_dataset

In [ ]:
train_dataset, val_dataset, test_dataset = split(dataset, 0.8, 0.1)

In [ ]:
!wandb login

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [ ]:
import wandb
from wandb.integration.keras import WandbMetricsLogger

In [ ]:
wandb.init(project = 'my-project', entity = 'shahrozhanif32-university-of-engineering-and-technology-')

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: shahrozhanif32 (shahrozhanif32-university-of-engineering-and-technology-). Use `wandb login --relogin` to force relogin


In [ ]:
wandb.config = {
    "learning_rate": 0.001,
    "batch_size": 32,
    "optimizer": "RMSprop",
    "im_size": 256,
    "kernel_size": 3
}
configuration = wandb.config

In [ ]:
resize_layer = tf.keras.layers.Resizing(configuration["im_size"], configuration["im_size"])
rescale_layer = tf.keras.layers.Rescaling(1/255)

def resize_rescale(image, label):
    image = resize_layer(image)
    image = rescale_layer(image)
    return image, label

train_dataset = train_dataset.map(resize_rescale)
val_dataset = val_dataset.map(resize_rescale)

test_dataset = test_dataset.map(resize_rescale)

In [ ]:
train_dataset = train_dataset.shuffle(buffer_size = 162).batch(configuration["batch_size"])
val_dataset = val_dataset.shuffle(buffer_size = 162).batch(configuration["batch_size"])
test_dataset = test_dataset.batch(1)

In [ ]:
from keras.layers import Normalization, Dense, Input, Conv2D, MaxPool2D, Flatten, BatchNormalization, RandomRotation, RandomFlip

augment_layers = tf.keras.Sequential([RandomFlip(mode='horizontal_and_vertical'), RandomRotation(0.25)])

model = tf.keras.Sequential([Input(shape = (configuration["im_size"], configuration["im_size"], 3)), augment_layers ,BatchNormalization(), Conv2D(filters = 4, kernel_size = configuration["kernel_size"], strides = 1, padding = 'valid', activation = "relu", use_bias = False), BatchNormalization(), MaxPool2D(pool_size = 2, strides = 2), Conv2D(filters = 8, kernel_size = configuration["kernel_size"], strides = 1, padding = 'valid', activation = "relu", use_bias = False), BatchNormalization(), MaxPool2D(pool_size = 2, strides = 2), Conv2D(filters = 16, kernel_size = configuration["kernel_size"], strides = 1, padding = 'valid', activation = "relu", use_bias = False), BatchNormalization(), MaxPool2D(pool_size = 2, strides = 2), Flatten(), Dense(100, activation = "relu"), BatchNormalization(), Dense(10, activation = "relu"), BatchNormalization(), Dense(1, activation = "sigmoid")])

In [ ]:
from keras.losses import BinaryCrossentropy
from keras.optimizers import Adam, RMSprop
from keras.metrics import BinaryAccuracy, TruePositives, TrueNegatives, FalsePositives, FalseNegatives, Precision, Recall, AUC
metrics = [BinaryAccuracy(), TruePositives(), TrueNegatives(), FalsePositives(), FalseNegatives(), Precision(), Recall()]
optimizer = getattr(keras.optimizers, configuration["optimizer"])
optimizer = optimizer(learning_rate = configuration["learning_rate"])
model.compile(optimizer = optimizer, loss = BinaryCrossentropy(), metrics = metrics)

In [ ]:
history = model.fit(train_dataset, validation_data = val_dataset, epochs = 5, verbose = 1, callbacks = [WandbMetricsLogger()])